In [1]:
# Section 1 — Setup and Imports
from dotenv import load_dotenv
load_dotenv()

from typing import Literal
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END, MessagesState

In [ ]:
# Section 2 — Document Loading
loader = PyPDFLoader("../documents/evs_oil_price_shock.pdf")
raw_docs = loader.load()

print(f"Loaded {len(raw_docs)} pages")

Loaded 15 pages


In [ ]:
# Section 3 — Text Chunking
splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunks = splitter.split_documents(raw_docs)

print(f"Split into {len(chunks)} chunks")

Split into 51 chunks


In [4]:
# Section 4 — Embedding Model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [5]:
# in-memory only; re-running this cell re-embeds from scratch
vectorstore = Chroma(
    collection_name="agentic_rag",
    embedding_function=embeddings,
)

vectorstore.add_documents(documents=chunks)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store ready")

Vector store ready


In [6]:
# Section 6 — LLM
llm = ChatOpenAI(model="gpt-5-mini")

In [ ]:
# Section 7 — State Definition
class AgenticRAGState(MessagesState):
    query: str
    retrieved_docs: list[Document] | None
    generation: str
    needs_retrieval: bool

In [ ]:
# route_question node — classifies whether the query needs document retrieval
def route_question(state: AgenticRAGState) -> dict:
    prompt = (
        "Does the following question require retrieving information from a specialized document, "
        "or can you answer it from your own general knowledge?\n\n"
        f"Question: {state['query']}\n\n"
        "Respond with exactly one of: 'needs_retrieval' or 'no_retrieval'."
    )
    response = llm.invoke(prompt)
    needs_retrieval = "needs_retrieval" in response.content.lower()
    return {"needs_retrieval": needs_retrieval}

In [ ]:
# retrieve node — fetches relevant documents from the vector store
def retrieve(state: AgenticRAGState) -> dict:
    docs = retriever.invoke(state["query"])
    return {"retrieved_docs": docs}

In [ ]:
# generate node — produces the final answer, with or without retrieved context
def generate(state: AgenticRAGState) -> dict:
    query = state["query"]
    docs = state.get("retrieved_docs") or []

    if docs:
        context = "\n\n".join(doc.page_content for doc in docs)
        prompt = (
            f"Answer the question using only the context below.\n\n"
            f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
        )
    else:
        prompt = f"Answer the following question from your general knowledge.\n\nQuestion: {query}\n\nAnswer:"

    response = llm.invoke(prompt)
    return {"generation": response.content}

In [ ]:
# Section 8 — Graph Definition
# routing function: maps needs_retrieval bool to the next node name
def route_after_classification(state: AgenticRAGState) -> Literal["retrieve", "generate"]:
    return "retrieve" if state["needs_retrieval"] else "generate"

graph_builder = StateGraph(AgenticRAGState)

graph_builder.add_node("route_question", route_question)
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("generate", generate)

graph_builder.add_edge(START, "route_question")
graph_builder.add_conditional_edges("route_question", route_after_classification)
graph_builder.add_edge("retrieve", "generate")
graph_builder.add_edge("generate", END)

In [ ]:
graph = graph_builder.compile()

In [ ]:
graph

In [ ]:
# Section 9 — Query 1: domain-specific (should trigger retrieval)
domain_query = "What does the report say about EV adoption trajectories and oil demand displacement?"

result_domain = graph.invoke({"query": domain_query, "messages": []})

In [ ]:
# Section 9 — Query 2: general knowledge (should skip retrieval)
general_query = "What is the capital of France?"

result_general = graph.invoke({"query": general_query, "messages": []})

In [ ]:
# Section 10 — Output: Query 1 (domain-specific)
print("=== QUERY 1 (domain-specific) ===")
print(f"Query          : {domain_query}")
print(f"needs_retrieval: {result_domain['needs_retrieval']}")
retrieved = result_domain.get("retrieved_docs") or []
print(f"Retrieved docs : {len(retrieved)} docs")
print(f"\nGeneration:\n{result_domain['generation']}")

In [ ]:
# Section 10 — Output: Query 2 (general knowledge)
print("=== QUERY 2 (general knowledge) ===")
print(f"Query          : {general_query}")
print(f"needs_retrieval: {result_general['needs_retrieval']}")
print(f"Retrieved docs : {result_general.get('retrieved_docs')}")
print(f"\nGeneration:\n{result_general['generation']}")